In [ ]:
import numpy as np
import pandas as pd

# Load Data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
columns = ['age','sex','cp','trestbps','chol','fbs','restecg',
           'thalach','exang','oldpeak','slope','ca','thal','target']

df = pd.read_csv(url, header=None, names=columns, na_values='?')
df.dropna(inplace=True)
df['target'] = (df['target'] > 0).astype(int)

# Prepare
X = df.drop('target', axis=1).values.astype(float)
y = df['target'].values.astype(float)

# Split
np.random.seed(42)
idx   = np.random.permutation(len(X))
split = int(0.8 * len(X))
X_train, X_test = X[idx[:split]], X[idx[split:]]
y_train, y_test = y[idx[:split]], y[idx[split:]]

# Normalize
mean = X_train.mean(axis=0)
std  = X_train.std(axis=0)
X_train = (X_train - mean) / std
X_test  = (X_test  - mean) / std

# Add Bias
X_train = np.hstack([np.ones((len(X_train), 1)), X_train])
X_test  = np.hstack([np.ones((len(X_test),  1)), X_test])

# Sigmoid
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# Train
weights = np.zeros(X_train.shape[1])
for i in range(1000):
    pred     = sigmoid(X_train @ weights)
    gradient = X_train.T @ (pred - y_train) / len(y_train)
    weights -= 0.1 * gradient

# Evaluate
y_pred   = (sigmoid(X_test @ weights) >= 0.5).astype(int)
accuracy = np.mean(y_pred == y_test) * 100
print(f"Accuracy: {accuracy:.2f}%")